# **Bronze Layer** -- ingest data to Databricks

In [0]:
import requests,json
from pyspark.sql import SparkSession

#defining spark session
Spark = SparkSession.builder.appName("JobETL_Bronze").getOrCreate()

#defining API key
API_KEY = "c65fa8831d153050aca77437b0a67a8ed519e9558b72f7b1ceb7105c64e0f42e"

roles = ["Data Engineer", "Python Developer", "ETL Developer", "Pyspark Developer", "Data Analyst"]
location="India"
all_jobs=[]

#running over a loop for the roles
for role in roles:
    params={
        "engine":"google_jobs",
        "q":role,
        "location":location,
        "api_key":API_KEY
    }
    print("✅Reading 🔴Live data from google Jobs API")
    response = requests.get("https://serpapi.com/search.json", params=params)
    jobs = response.json().get("jobs_results",[])

    for job in jobs:
        job["search_role"] = role
    all_jobs.extend(jobs)
bronze_df = spark.createDataFrame(all_jobs)
bronze_df.write.mode("append").option("mergeSchema", "true").saveAsTable("jobs_bronze")
print("✅Data written to bronze layer")




In [0]:
#display(bronze_df)

## **SILVER LAYER** - Cleaning and structuring the data

In [0]:
from pyspark.sql.functions import *

spark = SparkSession.builder.appName("JobETL_Silver").getOrCreate()
#reading bronze layer data
print("✅Reading data from bronze layer -- silver layer")
df = spark.read.table("jobs_bronze")

# cleaning and structuring
silver_df = df.selectExpr(
    "title",
    "company_name",
    "location",
    "description",
    "detected_extensions.posted_at as posted_at",
    "job_id",
    "search_role"
).dropna(subset=['title','company_name','location'])


In [0]:
display(silver_df)

In [0]:
silver_df = silver_df.withColumn("company_name", trim(upper(col("company_name"))))
silver_df = silver_df.dropna(subset=['description','posted_at','job_id'])
silver_df.write.mode("append").option("mergeSchema", "true").saveAsTable("jobs_silver")
print("✅Data written to the silver layer")
#display(silver_df)

## **GOLDEN LAYER  -- Generate KPI Tables**

In [0]:
from pyspark.sql.functions import *
spark = SparkSession.builder.appName("JobETL_Gold").getOrCreate()
#reading silver layer data
print("✅Reading data from Silver layer -- Gold Layer")
df = spark.read.table("jobs_silver")

#KPI 1  -- Top companies
top_companies = df.groupBy("company_name").count().orderBy(col("count").desc())
#display(top_companies)

#KPI 2 -- Top Cities
top_cities = df.groupBy("location").agg(count("*").alias("job_count")).orderBy(col("job_count").desc())
#display(top_cities)

top_companies.write.mode("append").option("mergeSchema", "true").saveAsTable("Jobs_Gold")

GLODEN LAYER METHOD2



In [0]:
from pyspark.sql.functions import *
spark = SparkSession.builder.appName("JobETL_Gold").getOrCreate()
#reading silver layer data
gold_df = spark.read.table("jobs_silver")

gold_df.write.mode("append").saveAsTable("JobsAggregator")
print("✅Data successfully return to Table")

In [0]:
%sql

select * from JobsAggregator;

In [0]:
print("😍Congrajulations !!!.. FULL ETL JobAggregator project completed and LIVE!!!")

In [0]:
%sql

describe history JobsAggregator